# Lab type: review
# Course: ML203 — Unsupervised Learning & Clustering
# Lesson: Hierarchical Clustering
# Task: Evaluate the hierarchical clustering implementation.
Answer the judgment questions.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
import matplotlib.pyplot as plt

## Step 1: Generate Data

In [ ]:
np.random.seed(42)

# Create hierarchical structure (nested clusters)
cluster_1a = np.random.normal([2, 2], 0.3, (20, 2))
cluster_1b = np.random.normal([3, 3], 0.3, (20, 2))
cluster_2 = np.random.normal([8, 8], 0.5, (40, 2))

X = np.vstack([cluster_1a, cluster_1b, cluster_2])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Data shape: {X_scaled.shape}")

## Step 2: Review Linkage Methods

In [ ]:
# Review these linkage methods — when would you use each one?
linkage_methods = ['ward', 'complete', 'average', 'single']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, method in enumerate(linkage_methods):
    Z = linkage(X_scaled, method=method)
    dendrogram(Z, ax=axes[idx], no_labels=True) # Corrected line
    axes[idx].set_title(f'Linkage: {method}')
    axes[idx].set_ylabel('Distance')

plt.tight_layout()
plt.show()

**Question:** The dendrograms look different for each linkage method. Which would you trust more? Why might `ward` behave differently from `single` linkage?

<details>
<summary>🔑 Reveal answer — Q1</summary>

**Trust ward linkage more** for most practical use cases. Ward minimises within-cluster variance at each merge step, producing compact, similarly-sized clusters that are easier to interpret and more resistant to noise.

**Why single linkage behaves differently:** Single linkage uses the minimum distance between any two points in the merging clusters. This makes it sensitive to chains of closely-spaced points — a few bridging observations can cause two large, distinct groups to merge early, producing elongated "spaghetti" clusters that don't represent real groupings.

**When single linkage is appropriate:** Only when you specifically need to detect connectivity — for example, manifold-shaped clusters or chains of related observations where nearest-neighbour paths matter more than cluster compactness.

</details>

## Step 3: Review Dendrogram Cutting

In [ ]:
# Review this code — is the cutting strategy reasonable?
Z = linkage(X_scaled, method='ward')

# Cut the dendrogram at different heights
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Full dendrogram
dendrogram(Z, ax=axes[0], no_labels=True)
axes[0].axhline(y=5, color='r', linestyle='--', label='Cut at height=5 (k=3)')
axes[0].axhline(y=8, color='g', linestyle='--', label='Cut at height=8 (k=2)')
axes[0].set_ylabel('Distance')
axes[0].legend()
axes[0].set_title('Dendrogram with Cut Lines')

# Plot 2: Cluster assignments for different cuts
labels_k3 = fcluster(Z, t=5, criterion='distance')
labels_k2 = fcluster(Z, t=8, criterion='distance')

print(f"Number of clusters at height=5: {len(set(labels_k3))}")
print(f"Number of clusters at height=8: {len(set(labels_k2))}")

**Question:** How do you decide where to cut the dendrogram? What objective or business constraint would guide your choice?

<details>
<summary>🔑 Reveal answer — Q2</summary>

**Objective guide:** Look for the tallest vertical line in the dendrogram that is not crossed by any horizontal cut. A large vertical gap indicates a merge step with high dissimilarity — a natural boundary where clusters are far apart. Cut just below that gap.

**Business constraint:** If the use case calls for a specific number of segments (e.g., four customer tiers), cut at the level that produces four clusters and verify each is interpretable and actionable.

**Do not:** Cut at a fixed height without examining what it produces. The dendrogram tells you the merging sequence; you bring the domain knowledge about what granularity is useful.

</details>

## Step 4: Review Hierarchical Structure Interpretation

In [ ]:
# Review this code — what does it tell you?
# A dendrogram shows structure, not just the final clustering

# Suppose you see this in the dendrogram:
# - Two clusters merge at distance ~2.5
# - These two merge into the final cluster at distance ~8.5
# - One outlier branch stays separate until height ~10

# This suggests:
print("Interpretation:")
print("1. There are 2 main natural clusters (merge at 2.5)")
print("2. A big gap at distance 8.5 suggests those 2 are truly separate")
print("3. The outlier at 10 might be noise or a very small cluster")


**Question:** How is hierarchical clustering different from k-means in terms of what it tells you about your data's structure?

<details>
<summary>🔑 Reveal answer — Q3</summary>

**K-means:** Requires a fixed k upfront and delivers only that one partition. You learn nothing about whether a different k would be more natural, or whether some clusters are nested inside others.

**Hierarchical clustering:** Builds a complete merge tree, letting you explore structure at every scale simultaneously. You can retroactively choose k by cutting the dendrogram at any level, which is especially valuable early in analysis when the right k is unknown.

**When hierarchy adds the most value:** When you suspect nested structure (e.g., broad product categories subdividing into specific types), or when you want to present multiple granularities to stakeholders without re-running the algorithm. The tradeoff is scalability — hierarchical methods are O(n²) or worse and become impractical for large datasets where k-means is preferred.

</details>

## Step 5: Compare Clustering Methods

In [ ]:
# You've now seen three clustering algorithms
# Here's a quick comparison:

comparison = pd.DataFrame({
    'Algorithm': ['K-Means', 'DBSCAN', 'Hierarchical'],
    'Requires k': ['Yes', 'No (has eps)', 'No, cut later'],
    'Non-spherical': ['No', 'Yes', 'Yes'],
    'Gives hierarchy': ['No', 'No', 'Yes'],
    'Best for': ['Spherical clusters', 'Arbitrary shapes', 'Exploring structure']
})

print(comparison.to_string(index=False))

<details>
<summary>🔑 Reveal summary answers</summary>

1. **Linkage choice:** Ward produces compact, interpretable clusters and is the reliable default; single linkage is for connectivity-based or chain-shaped structures only.
2. **Cutting the dendrogram:** Cut at the tallest gap (highest dissimilarity jump); validate that the resulting clusters make sense in your domain.
3. **Hierarchy vs k-means:** Hierarchical methods reveal structure at all levels simultaneously; k-means delivers one fixed partition — use hierarchical for exploration when k is unknown or nested structure is expected.

</details>

## Summary
Hierarchical clustering is powerful for exploration:
- **Dendrograms** show cluster structure at all levels, not just one k
- **Linkage method** affects how clusters merge (use ward for structure, single for connectivity)
- **Cutting the dendrogram** is guided by domain knowledge and the visual gaps in the dendrogram

**Next lesson:** PCA
— reducing dimensions while preserving structure.